# Task 2: Time Series Forecasting — ARIMA & LSTM

**GMF Investments — TSLA Price Forecasting**

Train: 2015–2024 | Test: 2025–2026

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings, os
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded!')

## 1. Load Data & Train/Test Split

In [ ]:
import yfinance as yf

# Load or fetch prices
csv_path = '../data/processed/prices.csv'
if os.path.exists(csv_path):
    prices = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    raw = yf.download(['TSLA','BND','SPY'], start='2015-01-01', end='2026-06-30', auto_adjust=True)
    prices = raw['Close'].ffill().bfill()

tsla = prices['TSLA'].dropna()

# Chronological split — train 2015-2024, test 2025-2026
train = tsla[tsla.index < '2025-01-01']
test  = tsla[tsla.index >= '2025-01-01']

print(f'TSLA total: {len(tsla):,} days')
print(f'Train: {train.index[0].date()} to {train.index[-1].date()} ({len(train):,} days)')
print(f'Test:  {test.index[0].date()} to {test.index[-1].date()} ({len(test):,} days)')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train, color='steelblue', label='Train (2015–2024)', linewidth=1)
ax.plot(test.index, test, color='coral', label='Test (2025–2026)', linewidth=1.5)
ax.axvline(pd.Timestamp('2025-01-01'), color='black', linestyle='--', alpha=0.7)
ax.set_title('TSLA Train/Test Split', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. ARIMA Model

### ACF/PACF to identify (p, d, q)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# First difference (d=1) to achieve stationarity
train_diff = train.diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_acf(train_diff, lags=40, ax=axes[0,0], title='ACF — Differenced TSLA')
plot_pacf(train_diff, lags=40, ax=axes[0,1], title='PACF — Differenced TSLA')
plot_acf(train, lags=40, ax=axes[1,0], title='ACF — Raw TSLA (non-stationary)')
plot_pacf(train, lags=40, ax=axes[1,1], title='PACF — Raw TSLA (non-stationary)')

plt.suptitle('ACF/PACF Analysis for ARIMA Parameter Selection', fontweight='bold')
plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('ACF/PACF plots saved!')

In [ ]:
# Auto ARIMA to find best parameters
from pmdarima import auto_arima

print('Running auto_arima (this may take 2-3 minutes)...')
auto_model = auto_arima(
    train,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    d=1,
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore',
    information_criterion='aic',
    n_jobs=-1
)
print(f'Best ARIMA order: {auto_model.order}')
print(f'AIC: {auto_model.aic():.2f}')
print(auto_model.summary())

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

p, d, q = auto_model.order
print(f'Training ARIMA({p},{d},{q})...')

arima_model = ARIMA(train, order=(p, d, q))
arima_fit   = arima_model.fit()

# Forecast on test period
arima_forecast = arima_fit.forecast(steps=len(test))
arima_forecast.index = test.index

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100

arima_mae  = mean_absolute_error(test, arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test, arima_forecast))
arima_mape = mape(test.values, arima_forecast.values)

print(f'\nARIMA({p},{d},{q}) Test Metrics:')
print(f'  MAE:  {arima_mae:.4f}')
print(f'  RMSE: {arima_rmse:.4f}')
print(f'  MAPE: {arima_mape:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-120:].index, train[-120:], color='steelblue', label='Training data', linewidth=1)
ax.plot(test.index, test, color='black', label='Actual (test)', linewidth=1.5)
ax.plot(arima_forecast.index, arima_forecast, color='#e74c3c',
        linestyle='--', label=f'ARIMA({p},{d},{q}) Forecast', linewidth=1.5)
ax.axvline(pd.Timestamp('2025-01-01'), color='grey', linestyle=':', alpha=0.7)
ax.set_title(f'ARIMA({p},{d},{q}) — TSLA Forecast vs Actual', fontweight='bold', fontsize=13)
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. LSTM Model

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tf.random.set_seed(42)

WINDOW = 60  # Use last 60 days to predict next day

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train.values.reshape(-1,1))

# Create sequences
def create_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, WINDOW)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')

In [ ]:
# Build LSTM architecture
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

print('Training LSTM...')
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print('Training complete!')

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('LSTM Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.savefig('lstm_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LSTM test prediction
# Prepare test input: need last WINDOW days of train + all test days
all_data = pd.concat([train, test])
all_scaled = scaler.transform(all_data.values.reshape(-1,1))

# Build test sequences
test_start_idx = len(train)
X_test = []
for i in range(test_start_idx, len(all_scaled)):
    X_test.append(all_scaled[i-WINDOW:i, 0])
X_test = np.array(X_test).reshape(-1, WINDOW, 1)

# Predict
lstm_pred_scaled = model.predict(X_test, verbose=0)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
lstm_forecast = pd.Series(lstm_pred, index=test.index)

# Metrics
lstm_mae  = mean_absolute_error(test, lstm_forecast)
lstm_rmse = np.sqrt(mean_squared_error(test, lstm_forecast))
lstm_mape = mape(test.values, lstm_forecast.values)

print(f'LSTM Test Metrics:')
print(f'  MAE:  {lstm_mae:.4f}')
print(f'  RMSE: {lstm_rmse:.4f}')
print(f'  MAPE: {lstm_mape:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-120:].index, train[-120:], color='steelblue', label='Training data', linewidth=1)
ax.plot(test.index, test, color='black', label='Actual (test)', linewidth=1.5)
ax.plot(lstm_forecast.index, lstm_forecast, color='#9b59b6',
        linestyle='--', label='LSTM Forecast', linewidth=1.5)
ax.axvline(pd.Timestamp('2025-01-01'), color='grey', linestyle=':', alpha=0.7)
ax.set_title('LSTM — TSLA Forecast vs Actual', fontweight='bold', fontsize=13)
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': [f'ARIMA({p},{d},{q})', 'LSTM (64-32 units)'],
    'MAE':  [arima_mae, lstm_mae],
    'RMSE': [arima_rmse, lstm_rmse],
    'MAPE (%)': [arima_mape, lstm_mape],
})
print('=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))

# Combined forecast plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train[-180:].index, train[-180:], color='steelblue', label='Training', linewidth=1, alpha=0.8)
ax.plot(test.index, test, color='black', label='Actual', linewidth=2)
ax.plot(arima_forecast.index, arima_forecast, color='#e74c3c',
        linestyle='--', label=f'ARIMA — MAPE:{arima_mape:.1f}%', linewidth=1.5)
ax.plot(lstm_forecast.index, lstm_forecast, color='#9b59b6',
        linestyle='--', label=f'LSTM — MAPE:{lstm_mape:.1f}%', linewidth=1.5)
ax.axvline(pd.Timestamp('2025-01-01'), color='grey', linestyle=':', alpha=0.7, label='Train/Test split')
ax.set_title('Model Comparison: ARIMA vs LSTM — TSLA Forecast', fontweight='bold', fontsize=13)
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = 'LSTM' if lstm_mape < arima_mape else f'ARIMA({p},{d},{q})'
print(f'\nBest model: {best}')
print('Saving best model for Task 3 forecasting...')

# Save for Task 3
os.makedirs('../data/processed', exist_ok=True)
comparison.to_csv('../data/processed/model_comparison.csv', index=False)
test.to_csv('../data/processed/tsla_test.csv')

# Save LSTM model
os.makedirs('../models', exist_ok=True)
model.save('../models/lstm_tsla.keras')
import pickle
with open('../models/scaler.pkl','wb') as f:
    pickle.dump(scaler, f)
arima_fit.save('../models/arima_tsla.pkl')
print('Models saved!')

## 5. Model Selection Discussion

### Which model performed better and why:

The **LSTM model** typically achieves lower MAPE on TSLA data because:
- TSLA exhibits strong non-linear patterns (momentum, sentiment-driven spikes) that ARIMA's linear framework cannot capture
- LSTM's 60-day window captures medium-term momentum signals that simple AR terms miss
- LSTM learns from the full distribution of historical sequences, not just local autocorrelations

**However**, ARIMA has advantages:
- Interpretable coefficients and residual diagnostics
- Faster training and prediction
- Better uncertainty quantification via confidence intervals
- Preferred in regulated financial contexts (Basel II-style interpretability requirements)

**For GMF's purposes:** LSTM is selected as the primary forecasting model for Task 3, with ARIMA retained as the interpretable baseline for regulatory documentation.